In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/train.csv")

X = df.drop(columns=['target', 'ID_code'])
y = df['target']
X.head()


,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,-4.9200,5.7470,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,3.1468,8.0851,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,-4.9193,5.9525,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,-5.8609,8.2450,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,6.2654,7.6784,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [2]:
# Row-wise statistical features
X['mean'] = X.mean(axis=1)
X['std'] = X.std(axis=1)
X['min'] = X.min(axis=1)
X['max'] = X.max(axis=1)

# Range feature
X['range'] = X['max'] - X['min']

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28036\848551123.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['mean'] = X.mean(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28036\848551123.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['std'] = X.std(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_28036\848551123.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.6).astype(int)  # best threshold

print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

F1: 0.4669692385274836
ROC-AUC: 0.8640416801484517


d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 🧪 Feature Engineering Experiments

### Objective
To improve model performance by creating new features from existing anonymized numerical data.

---

### Approach

Since the dataset contains anonymized features, domain-specific feature engineering is not possible. Instead, statistical features were created for each sample:

- Mean
- Standard Deviation (std)
- Minimum value
- Maximum value
- Range (max - min)

---

### Motivation

- Raw features lack interpretability
- Statistical summaries can capture overall patterns and variability
- Helps model understand global behavior of each sample

---

### Results

| Stage | F1-score | ROC-AUC |
|------|----------|--------|
| Baseline | ~0.37 | ~0.85 |
| + Class Weight | ~0.41 | ~0.85 |
| + Threshold Tuning | ~0.45 | ~0.85 |
| + Feature Engineering | **~0.47** | **~0.864** |

---

### Key Observations

- Feature engineering improved both F1-score and ROC-AUC
- Model performance improved without changing model complexity
- Statistical features provided meaningful additional information

---

### Insight

- Even without feature meaning, useful patterns can be extracted
- Statistical aggregation is effective for high-dimensional tabular data
- Feature engineering has a larger impact than simple model tuning

---

### Conclusion

- Feature engineering significantly improved classification performance
- These features will be retained in future experiments
- Further improvements will explore feature selection and dimensionality reduction